# Clase 191 — Synthetic Control Method

Abadie et al.: para una unidad tratada (ej: California aprobó Prop99, ¿efecto en consumo de tabaco?), construir un **control sintético** como combinación convexa de unidades no tratadas que matchea el pre-tratamiento. El gap post = efecto causal.
Requiere: `pip install numpy scipy matplotlib` (opcional `pysyncon`).

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N_control = 20
T_pre, T_post = 10, 10
T = T_pre + T_post

# Generamos N_control trayectorias con tendencia común + ruido
trend = np.linspace(50, 60, T)
controls = trend[None, :] + rng.normal(0, 1, (N_control, 1)) * np.arange(T)[None, :] * 0.1 \
           + rng.normal(0, 2, (N_control, T))
# Unidad tratada: combinación de 3 controles + ruido en pre, + efecto en post
true_weights = np.zeros(N_control); true_weights[[2, 7, 13]] = [0.4, 0.35, 0.25]
treated_pre  = controls[:, :T_pre].T @ true_weights + rng.normal(0, 0.5, T_pre)
treated_post = controls[:, T_pre:].T @ true_weights + rng.normal(0, 0.5, T_post)
TRUE_EFFECT = 5.0
treated_post += TRUE_EFFECT  # efecto aditivo desde t=T_pre
treated = np.concatenate([treated_pre, treated_post])
print(f'Treated shape: {treated.shape}  |  Controls: {controls.shape}  |  TRUE effect post = {TRUE_EFFECT}')

## Optimización: pesos convexos
$\min_w \| Y^{treated}_{pre} - W Y^{control}_{pre}\|^2$ s.a. $w_j \ge 0$, $\sum w_j = 1$.

In [ ]:
def fit_synthetic(treated_pre, controls_pre):
    n = controls_pre.shape[0]
    def loss(w):
        synth = controls_pre.T @ w
        return np.sum((treated_pre - synth)**2)
    w0 = np.ones(n) / n
    cons = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
    bounds = [(0.0, 1.0)] * n
    res = minimize(loss, w0, bounds=bounds, constraints=cons, method='SLSQP')
    return res.x

w_hat = fit_synthetic(treated_pre, controls[:, :T_pre])
print('pesos no-cero (top):', sorted([(i, round(w,3)) for i,w in enumerate(w_hat) if w>0.01], key=lambda x:-x[1]))
print('verdaderos        :', [(i, round(w,3)) for i,w in enumerate(true_weights) if w>0])

## Construir synthetic control y comparar

In [ ]:
synth = controls.T @ w_hat    # shape (T,)
gap = treated - synth
print(f'Gap pre  (mean abs) = {np.mean(np.abs(gap[:T_pre])):.3f}')
print(f'Gap post (mean)     = {gap[T_pre:].mean():.3f}  (TRUE effect = {TRUE_EFFECT})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(treated, label='Treated (real)', color='C3', lw=2)
ax.plot(synth,   label='Synthetic',      color='C0', lw=2, ls='--')
ax.axvline(T_pre-0.5, color='k', ls=':', alpha=0.6)
ax.set_title('Treated vs Synthetic'); ax.legend(); ax.set_xlabel('t')
ax = axes[1]
ax.plot(gap, color='C2', lw=2)
ax.axhline(0, color='k', ls=':')
ax.axvline(T_pre-0.5, color='k', ls=':', alpha=0.6)
ax.set_title('Gap = Treated - Synthetic'); ax.set_xlabel('t')
plt.tight_layout(); plt.show()

## Inferencia: placebo tests
Aplicamos el mismo método a cada unidad de control *como si* fuera la tratada.
Si el gap del tratamiento es **mucho mayor** que la distribución de gaps placebo → evidencia de efecto.

In [ ]:
placebo_gaps = []
for i in range(N_control):
    fake_treated = controls[i]
    fake_controls = np.delete(controls, i, axis=0)
    w_p = fit_synthetic(fake_treated[:T_pre], fake_controls[:, :T_pre])
    synth_p = fake_controls.T @ w_p
    placebo_gaps.append(fake_treated - synth_p)
placebo_gaps = np.array(placebo_gaps)

plt.figure(figsize=(10, 4))
for g in placebo_gaps:
    plt.plot(g, color='grey', alpha=0.4, lw=1)
plt.plot(gap, color='C3', lw=2.5, label='Treated gap')
plt.axhline(0, color='k', ls=':'); plt.axvline(T_pre-0.5, color='k', ls=':', alpha=0.6)
plt.legend(); plt.title('Gap real vs placebo gaps'); plt.show()

## Pseudo p-value (rank-based)
Ratio post/pre RMSE: si el del tratado es más alto que la mayoría de placebos → significativo.

In [ ]:
def rmse_ratio(gap_arr, T_pre):
    pre  = np.sqrt(np.mean(gap_arr[:T_pre]**2))
    post = np.sqrt(np.mean(gap_arr[T_pre:]**2))
    return post / pre if pre > 0 else np.inf

ratios = np.array([rmse_ratio(g, T_pre) for g in placebo_gaps])
treated_ratio = rmse_ratio(gap, T_pre)
rank = (np.sum(ratios >= treated_ratio) + 1) / (len(ratios) + 1)
print(f'Ratio post/pre (treated)  = {treated_ratio:.2f}')
print(f'Ratios placebo (mediana)  = {np.median(ratios):.2f}')
print(f'Pseudo p-value (rank)     = {rank:.3f}  (low = effect significant)')

## pysyncon (opcional)
API real para producción — exige DataFrames con id_unit, time, outcome, predictors.

In [ ]:
try:
    import pysyncon
    print('pysyncon disponible — ver pysyncon.Synth y pysyncon.AugSynth')
    print('Para producción suele convenir: pesos + matching de covariates predictoras.')
except ImportError:
    print('pysyncon no instalado; `pip install pysyncon` para API de alto nivel.')

## Takeaways
1. **Synthetic control** = combinación convexa de unidades no tratadas que matchea el pre-tratamiento.
2. Funciona con **una sola unidad tratada** — caso típico: políticas a nivel país/estado.
3. Inferencia: **placebo tests** + ratio RMSE post/pre.
4. Asunciones: que la combinación lineal sea apropiada y que no haya spillovers a controles.
5. Extensiones modernas: Augmented SC (Ben-Michael 2021), Generalized SC, `pysyncon`.